# Stock Price Prediction — XGBoost + LSTM
29 tickers, 5-min OHLCV bars (2020–2025), temporal train/test split, GPU training.

In [ ]:
import os, warnings, subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

## Download Data

In [ ]:
BASE_URL = 'https://8222dad36c757d4e-184-67-221-54.serveousercontent.com'
TICKERS = [
    'AAPL','AMD','AMZN','BABA','BAC','BA','C','CSCO','CVX','DIS',
    'F','GE','GOOGL','IBM','INTC','JNJ','JPM','KO','MCD','META',
    'MSFT','NFLX','NVDA','PFE','T','TSLA','VZ','WMT','XOM'
]

os.makedirs('stock_data', exist_ok=True)
data = {}
for ticker in TICKERS:
    path = f'stock_data/{ticker}.csv'
    if not os.path.exists(path) or os.path.getsize(path) < 1024:
        subprocess.run(
            ['wget', '-q', '-c', '--tries=5', '--timeout=60',
             '--no-check-certificate', '-O', path, f'{BASE_URL}/{ticker}.csv'],
            check=True
        )
    data[ticker] = pd.read_csv(path, parse_dates=['date'])
    print(f'{ticker}: {len(data[ticker])} rows')

# Merge all tickers into one DataFrame
prices = pd.concat(
    [df.assign(ticker=ticker)[['ticker','date','open','high','low','close','volume']]
     for ticker, df in data.items()],
    ignore_index=True
)
prices = prices.sort_values(['ticker','date']).reset_index(drop=True)
print(f'\nTotal rows: {len(prices):,} | Tickers: {prices["ticker"].nunique()}')

## Feature Engineering

In [ ]:
def create_features(prices: pd.DataFrame, horizon: int = 5):
    df = prices.copy().sort_values(['ticker', 'date']).reset_index(drop=True)
    g = df.groupby('ticker', group_keys=False)

    target_col = f'target_{horizon}d'
    df[target_col] = g['close'].shift(-horizon) / df['close'] - 1.0

    df['ret_1']     = g['close'].pct_change(1)
    df['ret_5']     = g['close'].pct_change(5)
    df['ret_10']    = g['close'].pct_change(10)
    df['ret_20']    = g['close'].pct_change(20)
    df['log_ret_1'] = np.log1p(df['ret_1'])
    df['hl_spread'] = (df['high'] - df['low']) / df['close'].replace(0, np.nan)
    df['oc_spread'] = (df['close'] - df['open']) / df['open'].replace(0, np.nan)

    for w in [10, 20, 50, 100]:
        df[f'sma_{w}']      = g['close'].transform(lambda s: s.rolling(w).mean())
        df[f'ema_{w}']      = g['close'].transform(lambda s: s.ewm(span=w, adjust=False).mean())
        df[f'dist_sma_{w}'] = df['close'] / df[f'sma_{w}'] - 1.0
        df[f'dist_ema_{w}'] = df['close'] / df[f'ema_{w}'] - 1.0

    df['ema_12']      = g['close'].transform(lambda s: s.ewm(span=12, adjust=False).mean())
    df['ema_26']      = g['close'].transform(lambda s: s.ewm(span=26, adjust=False).mean())
    df['macd']        = df['ema_12'] - df['ema_26']
    df['macd_signal'] = g['macd'].transform(lambda s: s.ewm(span=9, adjust=False).mean())
    df['macd_hist']   = df['macd'] - df['macd_signal']

    for w in [10, 20, 60]:
        df[f'volatility_{w}'] = g['ret_1'].transform(lambda s: s.rolling(w).std())
        roll_std  = g['close'].transform(lambda s: s.rolling(w).std())
        roll_mean = g['close'].transform(lambda s: s.rolling(w).mean())
        df[f'bb_{w}_width'] = (4.0 * roll_std) / roll_mean

    delta    = g['close'].diff()
    avg_gain = delta.clip(lower=0).groupby(df['ticker']).transform(lambda s: s.rolling(14).mean())
    avg_loss = (-delta.clip(upper=0)).groupby(df['ticker']).transform(lambda s: s.rolling(14).mean())
    df['rsi_14'] = 100 - (100 / (1 + avg_gain / avg_loss.replace(0, np.nan)))

    df['vol_ma_20']    = g['volume'].transform(lambda s: s.rolling(20).mean())
    df['vol_ratio_20'] = df['volume'] / df['vol_ma_20']
    for w in [5, 20, 60]:
        df[f'vol_change_{w}'] = g['volume'].pct_change(w)

    df['rolling_max_10']   = g['high'].transform(lambda s: s.rolling(10).max())
    df['rolling_min_10']   = g['low'].transform(lambda s: s.rolling(10).min())
    df['dist_roll_max_10'] = df['close'] / df['rolling_max_10'] - 1.0
    df['dist_roll_min_10'] = df['close'] / df['rolling_min_10'] - 1.0

    enc = LabelEncoder()
    df['ticker_id'] = enc.fit_transform(df['ticker'].astype(str))

    feature_cols = [c for c in df.columns if c not in {'ticker', 'date', target_col}]
    dataset = df.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)

    float_cols = dataset.select_dtypes(include=['float64']).columns
    dataset[float_cols] = dataset[float_cols].astype('float32')
    dataset['ticker_id'] = dataset['ticker_id'].astype('int32')

    return dataset, feature_cols, target_col, enc

## Build Dataset & Split

In [ ]:
TARGET_HORIZON = 5  # bars ahead (5 bars = 25 min on 5-min data)

dataset, feature_cols, target_col, encoder = create_features(prices, TARGET_HORIZON)

# Temporal split — 80% train / 20% test, no leakage
split_date = dataset['date'].quantile(0.80)
train_df = dataset[dataset['date'] <= split_date]
test_df  = dataset[dataset['date'] >  split_date]

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test,  y_test  = test_df[feature_cols],  test_df[target_col]

print(f'Split date : {pd.Timestamp(split_date).date()}')
print(f'Train rows : {len(X_train):,}')
print(f'Test rows  : {len(X_test):,}')
print(f'Features   : {len(feature_cols)}')

## XGBoost Training

In [ ]:

# Clean: replace inf with nan, then drop any remaining nan rows
X_train_c = X_train.replace([np.inf, -np.inf], np.nan).dropna()
y_train_c  = y_train.loc[X_train_c.index]
X_test_c   = X_test.replace([np.inf, -np.inf], np.nan).dropna()
y_test_c   = y_test.loc[X_test_c.index]

print(f"Train: {len(X_train_c):,} rows | Test: {len(X_test_c):,} rows")
print(f"NaNs in train: {X_train_c.isna().sum().sum()} | NaNs in test: {X_test_c.isna().sum().sum()}")

xgb = XGBRegressor(
    n_estimators=3000,
    learning_rate=0.02,
    max_depth=8,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='reg:squarederror',
    eval_metric='rmse',
    tree_method='hist',
    device='cuda',
    random_state=42,
    early_stopping_rounds=50,
)

xgb.fit(X_train_c, y_train_c, eval_set=[(X_test_c, y_test_c)], verbose=100)

pred_xgb = xgb.predict(X_test_c)
print(f'\nXGBoost RMSE : {np.sqrt(mean_squared_error(y_test_c, pred_xgb)):.6f}')
print(f'XGBoost MAE  : {mean_absolute_error(y_test_c, pred_xgb):.6f}')
print(f'Dir Accuracy : {np.mean(np.sign(pred_xgb) == np.sign(y_test_c)):.4f}')


## Feature Importance

In [ ]:
importance = pd.Series(xgb.feature_importances_, index=feature_cols)
top20 = importance.nlargest(20)

plt.figure(figsize=(10, 6))
top20.sort_values().plot(kind='barh')
plt.title('Top 20 Feature Importances (XGBoost)')
plt.tight_layout()
plt.show()

## LSTM Training

In [ ]:
import tensorflow as tf

SEQ_LEN = 30

train_df2 = dataset[dataset['date'] <= split_date].copy()
test_df2  = dataset[dataset['date'] >  split_date].copy()

# Scale from train stats only
mu = train_df2[feature_cols].mean()
sd = train_df2[feature_cols].std().replace(0, 1)
train_df2[feature_cols] = (train_df2[feature_cols] - mu) / sd
test_df2[feature_cols]  = (test_df2[feature_cols]  - mu) / sd

def make_sequences(df, feature_cols, target_col, seq_len=30):
    X, y = [], []
    for _, g in df.groupby('ticker', sort=False):
        g = g.sort_values('date')
        fv = g[feature_cols].to_numpy(np.float32)
        tv = g[target_col].to_numpy(np.float32)
        for i in range(seq_len, len(g)):
            X.append(fv[i-seq_len:i])
            y.append(tv[i])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_tr_seq, y_tr_seq = make_sequences(train_df2, feature_cols, target_col, SEQ_LEN)
X_te_seq, y_te_seq = make_sequences(test_df2,  feature_cols, target_col, SEQ_LEN)

lstm = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SEQ_LEN, len(feature_cols))),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(1),
])
lstm.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
lstm.fit(
    X_tr_seq, y_tr_seq,
    validation_data=(X_te_seq, y_te_seq),
    epochs=20,
    batch_size=256,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1,
)

pred_lstm = lstm.predict(X_te_seq, verbose=0).ravel()
print(f'LSTM RMSE : {np.sqrt(mean_squared_error(y_te_seq, pred_lstm)):.6f}')
print(f'LSTM MAE  : {mean_absolute_error(y_te_seq, pred_lstm):.6f}')
print(f'Dir Acc   : {np.mean(np.sign(pred_lstm) == np.sign(y_te_seq)):.4f}')